# 04 — 3D Swin (SwinUNETR encoder) with 5-fold CV

Whole-volume classifier: MONAI `SwinUNETR` encoder initialised from self-supervised CT pre-trained weights (not included — see README for the source), volumes resized to `64 × 96 × 96`, features of the last two pyramid levels pooled and concatenated into a linear head, focal loss with class weights, flip / Gaussian-noise / random crop-resize augmentation, differential learning rates (head vs backbone), threshold chosen with Youden's J on the validation fold. Trained on CPU (8 epochs per fold).

Note: the best epoch of each fold is chosen on that fold's validation AUC, so the reported numbers are optimistic (see README, *Limitations*).

Paths are Windows paths from the original machine (`D:\\NSCLC_PROJECT\\...`); change the path constants at the top of each notebook before running. Raw data and model weights are **not** included in this repository.

In [2]:
"""
آموزش Swin Transformer سه‌بعدی (3D Swin) برای طبقه‌بندی EGFR از روی حجم کامل CT
====================================================================================
- به‌جای ViT خام، از encoder سه‌بعدی SwinUNETR (یعنی SwinViT) استفاده می‌کنیم که
  می‌تونه با وزن‌های self-supervised pretrained (روی هزاران اسکن CT) initialize بشه.
- چون Z بین بیمارها خیلی متغیره، همه حجم‌ها به یه سایز ثابت (پیش‌فرض 64x96x96) resample میشن.
- حجم‌های resize‌شده روی دیسک کش میشن (پوشه volume_cache_3d) تا اجراهای بعدی سریع‌تر باشن.
- برای مقابله با imbalance: هم WeightedRandomSampler هم class-weighted loss استفاده میشه.
- انتخاب بهترین epoch فقط بر اساس AUC انجام میشه (چون threshold-independent هست).
- بعد از پایان train، threshold بهینه با Youden's J از روی val پیدا میشه (نه 0.5 ثابت).

نصب پیش‌نیاز:
    pip install monai torch torchvision scikit-learn pandas numpy

⚠️ قبل از اجرا: چک‌پوینت pretrained رو دانلود کنید و مسیرش رو در PRETRAINED_CKPT بذارید.
   منبع: https://github.com/Project-MONAI/research-contributions/tree/main/SwinUNETR/Pretrain
   (یا از MONAI Model Zoo چک‌پوینت SSL مربوط به SwinUNETR رو بگیرید)
   اگه چک‌پوینت ندارید، PRETRAINED_CKPT = None بذارید تا از صفر train بشه
   (ولی در اون صورت نتایج احتمالاً شبیه ViT خام ضعیف می‌مونه).
"""

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from monai.networks.nets import SwinUNETR
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, balanced_accuracy_score, roc_curve
)

# ==================== تنظیمات ====================
METADATA_CSV = r"D:\NSCLC_PROJECT\dataset_metadata.csv"
EXCLUDE_IDS = ["R01-091", "R01-133"]
CACHE_DIR = r"D:\NSCLC_PROJECT\volume_cache_3d"           # کش دیسک حجم‌های resize شده
TARGET_SHAPE = (64, 96, 96)             # (Z,H,W) بعد از resample
FEATURE_SIZE = 48                       # سایز پایه‌ی فیچرهای SwinUNETR (باید بر 32 بخش‌پذیر باشه)
PRETRAINED_CKPT = r"D:\NSCLC_PROJECT\model_swinvit.pt"   # مسیر چک‌پوینت SSL / یا None
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 12
LR_HEAD = 5e-4        # نرخ یادگیری برای head (تازه‌ساخته‌شده)
LR_BACKBONE = 1e-5    # نرخ یادگیری کوچیک‌تر برای backbone pretrained
FREEZE_BACKBONE_EPOCHS = 2   # چند epoch اول فقط head train بشه، بعد backbone هم باز بشه
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
RESULTS_DIR = r"D:\NSCLC_PROJECT"
# ===================================================

os.makedirs(CACHE_DIR, exist_ok=True)

NUM_WORKERS = 2
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count())
    NUM_WORKERS = 0
    BATCH_SIZE = 2
    NUM_EPOCHS = 8
    FREEZE_BACKBONE_EPOCHS = 2
    print(f"⚠️ اجرا روی CPU. BATCH_SIZE={BATCH_SIZE} | NUM_EPOCHS={NUM_EPOCHS} | threads={os.cpu_count()}")
    print("   تخمین زمان: هر fold حدود ۳۰-۶۰ دقیقه (بار اول کندتره چون کش دیسک خالیه).\n")

torch.manual_seed(SEED)
np.random.seed(SEED)


def get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps):
    """Warmup خطی + cosine decay -> جلوگیری از ناپایداری/collapse ترنسفورمر تو ابتدای آموزش."""
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class FocalLoss(nn.Module):
    """
    به‌جای CrossEntropy ساده: روی نمونه‌های سخت/اقلیت بیشتر تمرکز می‌کنه.
    با imbalance شدید (که اینجا داریم) معمولاً بهتر از CE + label smoothing عمل می‌کنه.

    نکته‌ی مهم: pt باید از روی cross-entropy خام (بدون وزن کلاس) محاسبه بشه، وگرنه
    وقتی alpha (وزن کلاس) خیلی کوچیک باشه (مثلاً 1/count)، ce مصنوعاً کوچیک می‌شه،
    pt به 1 نزدیک می‌شه، و (1-pt)^gamma کل loss رو تقریباً صفر می‌کنه -> مدل چیزی
    یاد نمی‌گیره (این همون باگی بود که loss رو صفر نشون می‌داد).
    """
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_raw = F.cross_entropy(logits, targets, reduction="none")  # بدون وزن -> pt درست
        pt = torch.exp(-ce_raw)
        focal_term = (1 - pt) ** self.gamma

        if self.alpha is not None:
            alpha_t = self.alpha[targets]           # وزن هر نمونه بر اساس کلاسش
            alpha_t = alpha_t / self.alpha.mean()    # نرمالایز -> میانگین وزن‌ها = ۱
            loss = alpha_t * focal_term * ce_raw
        else:
            loss = focal_term * ce_raw

        return loss.mean()


def load_and_resize_volume(file_path, clean_id):
    """حجم رو می‌خونه، به TARGET_SHAPE resample می‌کنه، و رو دیسک کش می‌کنه."""
    cache_path = os.path.join(CACHE_DIR, f"{clean_id}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    volume = np.load(file_path).astype(np.float32)              # (Z_orig, 380, 380)
    vol_t = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0)   # (1,1,Z,H,W)
    vol_resized = F.interpolate(vol_t, size=TARGET_SHAPE, mode="trilinear", align_corners=False)
    vol_resized = vol_resized.squeeze(0).squeeze(0).numpy()      # (64,96,96)

    np.save(cache_path, vol_resized)
    del volume, vol_t
    return vol_resized


def precompute_volumes(df):
    print("در حال بارگذاری/resize حجم‌های سه‌بعدی ...")
    cache = {}
    for i, (_, row) in enumerate(df.iterrows()):
        cache[row["clean_id"]] = load_and_resize_volume(row["file_path"], row["clean_id"])
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(df)} بیمار پردازش شد")
    print("تمام شد.\n")
    return cache


class CTVolumeDataset(Dataset):
    def __init__(self, df, volume_cache, augment=False, target_shape=None):
        self.rows = df.reset_index(drop=True)
        self.volume_cache = volume_cache
        self.augment = augment
        self.target_shape = target_shape  # (Z,H,W) - برای crop تصادفی لازمه

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows.iloc[i]
        vol = self.volume_cache[row["clean_id"]].copy()
        vol_t = torch.from_numpy(vol).unsqueeze(0)  # (1,64,96,96) تک‌کاناله

        if self.augment:
            if torch.rand(1).item() < 0.5:
                vol_t = torch.flip(vol_t, dims=[3])  # فلیپ افقی (چپ/راست ریه)
            if torch.rand(1).item() < 0.3:
                vol_t = torch.clamp(vol_t + torch.randn_like(vol_t) * 0.01, 0, 1)
            # --- random crop + pad: یه پنجره‌ی کوچیک‌تر رو در موقعیت تصادفی برمی‌داریم
            # و دوباره به سایز اصلی برمی‌گردونیم. تنوع مکانی بیشتری به مدل با دیتای
            # کم می‌ده و کمی شبیه zoom/shift augmentation عمل می‌کنه.
            if torch.rand(1).item() < 0.5 and self.target_shape is not None:
                vol_t = self._random_crop_resize(vol_t, self.target_shape, crop_ratio=0.85)

        return vol_t, int(row["label"]), row["clean_id"]

    @staticmethod
    def _random_crop_resize(vol_t, target_shape, crop_ratio=0.85):
        c, z, h, w = vol_t.shape
        cz, ch, cw = int(z * crop_ratio), int(h * crop_ratio), int(w * crop_ratio)
        z0 = torch.randint(0, max(1, z - cz + 1), (1,)).item()
        h0 = torch.randint(0, max(1, h - ch + 1), (1,)).item()
        w0 = torch.randint(0, max(1, w - cw + 1), (1,)).item()
        cropped = vol_t[:, z0:z0 + cz, h0:h0 + ch, w0:w0 + cw]
        cropped = cropped.unsqueeze(0)  # (1,1,cz,ch,cw)
        resized = F.interpolate(cropped, size=target_shape, mode="trilinear", align_corners=False)
        return resized.squeeze(0)


class Swin3DClassifier(nn.Module):
    """
    از encoder (SwinViT) داخل SwinUNETR فقط برای classification استفاده می‌کنیم.
    برخلاف نسخه‌ی قبلی که فقط عمیق‌ترین سطح هرم استفاده می‌شد، اینجا فیچرهای دو سطح
    آخر (میانی + عمیق) رو با هم global-average-pool و concat می‌کنیم -> هم جزئیات
    محلی‌تر هم context کلی‌تر رو در نظر می‌گیره، که برای دیتای کم معمولاً کمک می‌کنه.
    """
    def __init__(self, img_size, in_channels=1, feature_size=48, num_classes=2, dropout=0.2):
        super().__init__()
        # نکته: img_size به SwinUNETR پاس داده نمیشه چون نسخه‌ی نصب‌شده‌ی MONAI
        # دیگه این آرگومان رو در سازنده قبول نمی‌کنه (در نسخه‌های جدیدتر حذف شده).
        backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=1,          # فقط encoder لازمه، این مقدار بی‌اثره
            feature_size=feature_size,
            use_checkpoint=True,
        )
        self.swinViT = backbone.swinViT
        del backbone  # بقیه‌ی decoder رو نگه نمی‌داریم

        self.pool = nn.AdaptiveAvgPool3d(1)
        # سطح آخر: کانال feature_size*16 | سطح ماقبل‌آخر: کانال feature_size*8
        combined_channels = feature_size * 16 + feature_size * 8
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_channels, num_classes),
        )

    def forward(self, x):
        hidden_states = self.swinViT(x)   # لیستی از feature mapها در سطوح مختلف هرم
        feat_deep = hidden_states[-1]     # عمیق‌ترین سطح
        feat_mid = hidden_states[-2]      # سطح ماقبل‌آخر
        pooled = torch.cat([
            self.pool(feat_deep).flatten(1),
            self.pool(feat_mid).flatten(1),
        ], dim=1)
        return self.head(pooled)


def build_model():
    model = Swin3DClassifier(
        img_size=TARGET_SHAPE,
        in_channels=1,
        feature_size=FEATURE_SIZE,
        num_classes=2,
        dropout=0.2,
    )

    if PRETRAINED_CKPT is not None and os.path.exists(PRETRAINED_CKPT):
        ckpt = torch.load(PRETRAINED_CKPT, map_location="cpu")
        state_dict = ckpt.get("state_dict", ckpt)
        # این چک‌پوینت فقط خود swinViT رو (با DataParallel) train کرده -> کلیدها با
        # "module." شروع میشن، نه "swinViT.". فقط پیشوند "module." رو حذف می‌کنیم.
        # همچنین بین نسخه‌های مختلف MONAI اسم لایه‌های MLP عوض شده:
        # چک‌پوینت قدیمی: mlp.fc1 / mlp.fc2  ->  نسخه‌ی فعلی: mlp.linear1 / mlp.linear2
        # کلیدهای دیگه‌ای مثل norm/convTrans3d/rotation_head/contrastive_head مربوط به
        # head‌های کمکی self-supervised pretraining هستن و عمداً نادیده گرفته میشن.
        filtered = {}
        for k, v in state_dict.items():
            new_k = k[len("module."):] if k.startswith("module.") else k
            new_k = new_k.replace("mlp.fc1", "mlp.linear1").replace("mlp.fc2", "mlp.linear2")
            filtered[new_k] = v
        missing, unexpected = model.swinViT.load_state_dict(filtered, strict=False)
        matched = len(filtered) - len(unexpected)
        print(f"  [Pretrained] کلیدهای لود شده از چک‌پوینت: {matched} | "
              f"Missing: {len(missing)} | Unexpected: {len(unexpected)}")
        if missing:
            print(f"  --- نمونه‌ی کلیدهای Missing (تو مدل هستن ولی تو چک‌پوینت نیستن) ---")
            for k in list(missing)[:15]:
                print(f"    {k}")
        if unexpected:
            print(f"  --- نمونه‌ی کلیدهای Unexpected (تو چک‌پوینت هستن ولی مدل نمی‌شناسه) ---")
            for k in list(unexpected)[:15]:
                print(f"    {k}")
        if len(missing) > len(filtered) * 0.5:
            print("  ⚠️ هشدار: بیشتر کلیدها لود نشدن. چک کنید مسیر چک‌پوینت و ساختار state_dict درسته.")
        else:
            print("  ✅ اکثر وزن‌های pretrained با موفقیت لود شدن.")
    else:
        print("  ⚠️ چک‌پوینت pretrained پیدا نشد -> backbone از صفر مقداردهی میشه (random init).")

    return model.to(DEVICE)


def set_backbone_trainable(model, trainable: bool):
    for p in model.swinViT.parameters():
        p.requires_grad = trainable


def evaluate_patient_level(model, loader):
    model.eval()
    all_probs, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for vols, labels, ids in loader:
            vols = vols.to(DEVICE)
            logits = model(vols)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy() if torch.is_tensor(labels) else labels)
            all_ids.extend(ids)
    return pd.DataFrame({"clean_id": all_ids, "prob": all_probs, "label": all_labels})


def train_one_fold(train_df, val_df, volume_cache, fold_num):
    train_ds = CTVolumeDataset(train_df, volume_cache, augment=True, target_shape=TARGET_SHAPE)
    val_ds = CTVolumeDataset(val_df, volume_cache, augment=False)

    labels_arr = train_df["label"].values
    class_counts = np.bincount(labels_arr)
    class_weights = 1.0 / class_counts

    # --- WeightedRandomSampler برای balance کردن هر batch ---
    sample_weights = class_weights[labels_arr]
    sampler = WeightedRandomSampler(
        sample_weights, num_samples=len(sample_weights), replacement=True
    )
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_model()

    class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)

    # --- discriminative learning rate: backbone آروم‌تر، head سریع‌تر ---
    optimizer = torch.optim.AdamW([
        {"params": model.swinViT.parameters(), "lr": LR_BACKBONE},
        {"params": model.head.parameters(), "lr": LR_HEAD},
    ], weight_decay=1e-3)

    steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM_STEPS)
    total_steps = steps_per_epoch * NUM_EPOCHS
    warmup_steps = max(10, total_steps // 10)
    scheduler = get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)

    best_score, best_state = -1, None

    for epoch in range(NUM_EPOCHS):
        # --- progressive unfreezing: چند epoch اول فقط head train میشه ---
        set_backbone_trainable(model, trainable=(epoch >= FREEZE_BACKBONE_EPOCHS))

        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for i, (vols, labels, _) in enumerate(train_loader):
            vols, labels = vols.to(DEVICE), labels.to(DEVICE)
            logits = model(vols)
            loss = criterion(logits, labels) / GRAD_ACCUM_STEPS
            loss.backward()
            total_loss += loss.item() * vols.size(0) * GRAD_ACCUM_STEPS

            if (i + 1) % GRAD_ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        val_pred_df = evaluate_patient_level(model, val_loader)
        auc = roc_auc_score(val_pred_df["label"], val_pred_df["prob"]) if val_pred_df["label"].nunique() > 1 else 0.5
        bal_acc_05 = balanced_accuracy_score(val_pred_df["label"], (val_pred_df["prob"] >= 0.5).astype(int))

        # --- تغییر ۱: انتخاب best epoch فقط بر اساس AUC (threshold-independent) ---
        score = auc

        frozen_note = " [backbone frozen]" if epoch < FREEZE_BACKBONE_EPOCHS else ""
        print(f"  [Fold {fold_num}] Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_ds):.4f} | "
              f"val_AUC={auc:.3f} | val_bal_acc@0.5={bal_acc_05:.3f}{frozen_note}")

        if score > best_score:
            best_score = score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    final_df = evaluate_patient_level(model, val_loader)

    # --- تغییر ۲: threshold بهینه با Youden's J به‌جای 0.5 ثابت ---
    fpr, tpr, thresholds = roc_curve(final_df["label"], final_df["prob"])
    j_scores = tpr - fpr
    best_thresh = thresholds[np.argmax(j_scores)]
    preds = (final_df["prob"] >= best_thresh).astype(int)
    print(f"  [Fold {fold_num}] Threshold بهینه: {best_thresh:.3f}")

    metrics = {
        "fold": fold_num,
        "accuracy": accuracy_score(final_df["label"], preds),
        "balanced_accuracy": balanced_accuracy_score(final_df["label"], preds),
        "auc": roc_auc_score(final_df["label"], final_df["prob"]),
        "f1": f1_score(final_df["label"], preds, zero_division=0),
        "best_threshold": best_thresh,
    }
    return metrics, final_df


def main():
    df = pd.read_csv(METADATA_CSV)
    df = df[~df["clean_id"].isin(EXCLUDE_IDS)].reset_index(drop=True)
    print(f"تعداد بیمار پس از حذف نمونه‌های خراب: {len(df)}")

    volume_cache = precompute_volumes(df)

    all_metrics, oof_preds = [], []
    for fold_num in sorted(df["fold"].unique()):
        train_df = df[df["fold"] != fold_num]
        val_df = df[df["fold"] == fold_num]
        print(f"\n=== Fold {fold_num} | train={len(train_df)} | val={len(val_df)} ===")

        metrics, patient_preds = train_one_fold(train_df, val_df, volume_cache, fold_num)
        all_metrics.append(metrics)
        oof_preds.append(patient_preds)
        print(f"  >>> نتیجه Fold {fold_num}: acc={metrics['accuracy']:.3f} | "
              f"AUC={metrics['auc']:.3f} | F1={metrics['f1']:.3f} | "
              f"balanced_acc={metrics['balanced_accuracy']:.3f} | thresh={metrics['best_threshold']:.3f}")

    results_df = pd.DataFrame(all_metrics)
    results_df.to_csv(os.path.join(RESULTS_DIR, "swin3d_cv_results.csv"), index=False)
    pd.concat(oof_preds).to_csv(os.path.join(RESULTS_DIR, "swin3d_patient_predictions.csv"), index=False)

    print("\n========== خلاصه Cross-Validation (3D Swin) ==========")
    print(results_df[["fold", "accuracy", "balanced_accuracy", "auc", "f1", "best_threshold"]])
    print("\nمیانگین ± انحراف معیار:")
    for col in ["accuracy", "balanced_accuracy", "auc", "f1"]:
        print(f"  {col}: {results_df[col].mean():.3f} ± {results_df[col].std():.3f}")


if __name__ == "__main__":
    main()

⚠️ اجرا روی CPU. BATCH_SIZE=2 | NUM_EPOCHS=8 | threads=16
   تخمین زمان: هر fold حدود ۳۰-۶۰ دقیقه (بار اول کندتره چون کش دیسک خالیه).

تعداد بیمار پس از حذف نمونه‌های خراب: 170
در حال بارگذاری/resize حجم‌های سه‌بعدی ...
  20/170 بیمار پردازش شد
  40/170 بیمار پردازش شد
  60/170 بیمار پردازش شد
  80/170 بیمار پردازش شد
  100/170 بیمار پردازش شد
  120/170 بیمار پردازش شد
  140/170 بیمار پردازش شد
  160/170 بیمار پردازش شد
تمام شد.


=== Fold 0 | train=135 | val=35 ===
  [Pretrained] کلیدهای لود شده از چک‌پوینت: 126 | Missing: 0 | Unexpected: 8
  --- نمونه‌ی کلیدهای Unexpected (تو چک‌پوینت هستن ولی مدل نمی‌شناسه) ---
    norm.weight
    norm.bias
    convTrans3d.weight
    convTrans3d.bias
    rotation_head.weight
    rotation_head.bias
    contrastive_head.weight
    contrastive_head.bias
  ✅ اکثر وزن‌های pretrained با موفقیت لود شدن.
  [Fold 0] Epoch 1/8 | loss=0.1836 | val_AUC=0.303 | val_bal_acc@0.5=0.500 [backbone frozen]
  [Fold 0] Epoch 2/8 | loss=0.1777 | val_AUC=0.496 | val_bal_a